# Mini Research #2
# Can Simple Image Preprocessing Recover YOLO Object-Detection Performance After JPEG Compression?

**Research question:** Can simple image preprocessing techniques recover YOLO object-detection performance after JPEG compression?

**Dataset:** 20 selected COCO train2017 images  
**Ground truth:** 238 non-crowd objects  
**Detector:** pretrained Ultralytics YOLO26n  
**Conditions:** Original → Q70 → Q50 → Q30 → Q30 + preprocessing

This notebook documents the complete computational workflow. The model is **not retrained**; the same pretrained checkpoint and ground-truth annotations are used across conditions.

## 1. Experimental design

### Phase 1 — Compression effect
- Original
- JPEG Q70
- JPEG Q50
- JPEG Q30

### Phase 2 — Recovery at Q30
- Q30 + Sharpening
- Q30 + Denoising
- Q30 + 2× Lanczos Upscaling

### Main metrics
- Precision
- Recall
- mAP50
- mAP50-95

A per-image sanity analysis is included because the study contains only 20 images.

In [ ]:
# 2. Environment
!pip -q install ultralytics

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import cv2
from ultralytics import YOLO

print("Environment ready.")

In [ ]:
# 3. Paths and model
BASE_DIR = Path("/content/mini02_dataset")

IMAGE_DIRS = {
    "Original": BASE_DIR / "images",
    "Q70": BASE_DIR / "images_q70",
    "Q50": BASE_DIR / "images_q50",
    "Q30": BASE_DIR / "images_q30",
}

LABEL_DIR = BASE_DIR / "labels"

model = YOLO("yolo26n.pt")
print("Model loaded successfully.")

## 4. Dataset sanity checks

The prepared dataset contains 20 original images and 238 ground-truth bounding boxes.

In [ ]:
for name, path in IMAGE_DIRS.items():
    print(f"{name:10s}: {len(list(path.glob('*.jpg')))} images")

print(f"Ground Truth labels: {len(list(LABEL_DIR.glob('*.txt')))}")

In [ ]:
bad_images = []

for image_path in sorted(IMAGE_DIRS["Original"].glob("*.jpg")):
    try:
        with Image.open(image_path) as img:
            img.verify()
    except Exception as e:
        bad_images.append((image_path.name, str(e)))

print("Invalid images:", len(bad_images))
if bad_images:
    print(bad_images)

## 5. JPEG compression generation

The same 20 original images are encoded at quality factors 70, 50, and 30.

In [ ]:
for directory in [IMAGE_DIRS["Q70"], IMAGE_DIRS["Q50"], IMAGE_DIRS["Q30"]]:
    directory.mkdir(parents=True, exist_ok=True)

for image_path in sorted(IMAGE_DIRS["Original"].glob("*.jpg")):
    img = Image.open(image_path).convert("RGB")
    for quality, condition in [(70, "Q70"), (50, "Q50"), (30, "Q30")]:
        img.save(IMAGE_DIRS[condition] / image_path.name, "JPEG", quality=quality)

print("JPEG versions generated.")

## 6. Load ground truth

In [ ]:
def load_yolo_labels(label_path):
    labels = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:])
            labels.append({"class_id": class_id, "box": [x_center, y_center, width, height]})
    return labels

ground_truth = {}
for label_file in sorted(LABEL_DIR.glob("*.txt")):
    ground_truth[label_file.stem + ".jpg"] = load_yolo_labels(label_file)

print(f"Images with Ground Truth: {len(ground_truth)}")
print(f"Total Ground Truth boxes: {sum(len(v) for v in ground_truth.values())}")

## 7. YOLO inference on Original / Q70 / Q50 / Q30

A very low confidence threshold is used while collecting predictions so that low-confidence detections remain available for AP calculation.

In [ ]:
results_for_ap = defaultdict(list)

for condition, image_dir in IMAGE_DIRS.items():
    print(f"Running YOLO on: {condition}")
    for image_path in sorted(image_dir.glob("*.jpg")):
        result = model.predict(source=str(image_path), conf=0.001, verbose=False)[0]
        results_for_ap[condition].append({"image": image_path.name, "result": result})
    print(f"Completed: {len(list(image_dir.glob('*.jpg')))} images")

In [ ]:
# Qualitative check: image_01.jpg
image_name = "image_01.jpg"

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, condition in zip(axes, IMAGE_DIRS.keys()):
    item = next(x for x in results_for_ap[condition] if x["image"] == image_name)
    plotted = item["result"].plot()
    ax.imshow(plotted[:, :, ::-1])
    ax.set_title(condition)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. IoU utilities

In [ ]:
def xywhn_to_xyxy(box):
    xc, yc, w, h = box
    return [xc - w/2, yc - h/2, xc + w/2, yc + h/2]

def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    iw = max(0.0, x2 - x1)
    ih = max(0.0, y2 - y1)
    intersection = iw * ih

    area1 = max(0.0, box1[2]-box1[0]) * max(0.0, box1[3]-box1[1])
    area2 = max(0.0, box2[2]-box2[0]) * max(0.0, box2[3]-box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

## 9. Extract normalized predictions

In [ ]:
def extract_predictions(results_dict):
    predictions = {}

    for condition, items in results_dict.items():
        predictions[condition] = {}

        for item in items:
            image_name = item["image"]
            result = item["result"]
            image_predictions = []

            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)

                width = result.orig_shape[1]
                height = result.orig_shape[0]

                for box, score, class_id in zip(boxes, scores, classes):
                    x1, y1, x2, y2 = box
                    image_predictions.append({
                        "class_id": int(class_id),
                        "confidence": float(score),
                        "box": [x1/width, y1/height, x2/width, y2/height]
                    })

            predictions[condition][image_name] = image_predictions

    return predictions

predictions = extract_predictions(results_for_ap)
print("Prediction extraction completed.")

## 10. Fixed-threshold Precision and Recall

Confidence ≥ 0.25 and IoU ≥ 0.50 are used for the fixed-threshold TP/FP/FN analysis.

In [ ]:
def calculate_precision_recall(condition_predictions, ground_truth,
                                 conf_threshold=0.25, iou_threshold=0.50):
    TP = FP = FN = 0

    for image_name, gt_items in ground_truth.items():
        gt_boxes = [
            {"class_id": x["class_id"], "box": xywhn_to_xyxy(x["box"]), "matched": False}
            for x in gt_items
        ]

        preds = [
            p for p in condition_predictions.get(image_name, [])
            if p["confidence"] >= conf_threshold
        ]
        preds = sorted(preds, key=lambda x: x["confidence"], reverse=True)

        for pred in preds:
            best_iou = 0.0
            best_gt = None

            for gt in gt_boxes:
                if gt["matched"] or pred["class_id"] != gt["class_id"]:
                    continue

                iou = box_iou(pred["box"], gt["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt = gt

            if best_gt is not None and best_iou >= iou_threshold:
                TP += 1
                best_gt["matched"] = True
            else:
                FP += 1

        FN += sum(not gt["matched"] for gt in gt_boxes)

    precision = TP/(TP+FP) if TP+FP else 0.0
    recall = TP/(TP+FN) if TP+FN else 0.0
    return TP, FP, FN, precision, recall

pr_results = []
for condition in IMAGE_DIRS:
    TP, FP, FN, precision, recall = calculate_precision_recall(
        predictions[condition], ground_truth
    )
    pr_results.append({
        "Condition": condition, "TP": TP, "FP": FP, "FN": FN,
        "Precision": precision, "Recall": recall
    })

pr_df = pd.DataFrame(pr_results)
print(pr_df.to_string(index=False))

## 11. AP / mAP calculation

AP is computed separately for each class present in the subset. mAP50 is the macro-average at IoU 0.50; mAP50-95 is the macro-average over IoU thresholds 0.50–0.95.

In [ ]:
def compute_ap_for_class(condition_predictions, ground_truth, class_id, iou_threshold):
    gt_by_image = {}
    total_gt = 0

    for image_name, gt_items in ground_truth.items():
        gt_boxes = []
        for gt in gt_items:
            if gt["class_id"] == class_id:
                gt_boxes.append({"box": xywhn_to_xyxy(gt["box"]), "matched": False})
        gt_by_image[image_name] = gt_boxes
        total_gt += len(gt_boxes)

    if total_gt == 0:
        return np.nan

    all_predictions = []
    for image_name, preds in condition_predictions.items():
        for pred in preds:
            if pred["class_id"] == class_id:
                all_predictions.append({
                    "image": image_name,
                    "confidence": pred["confidence"],
                    "box": pred["box"]
                })

    all_predictions.sort(key=lambda x: x["confidence"], reverse=True)

    TP = np.zeros(len(all_predictions))
    FP = np.zeros(len(all_predictions))

    for i, pred in enumerate(all_predictions):
        best_iou = 0.0
        best_gt = None

        for gt in gt_by_image[pred["image"]]:
            if gt["matched"]:
                continue
            iou = box_iou(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt = gt

        if best_gt is not None and best_iou >= iou_threshold:
            TP[i] = 1
            best_gt["matched"] = True
        else:
            FP[i] = 1

    cumulative_tp = np.cumsum(TP)
    cumulative_fp = np.cumsum(FP)

    recalls = cumulative_tp / total_gt
    precisions = cumulative_tp / (cumulative_tp + cumulative_fp + 1e-16)

    mrec = np.concatenate(([0.0], recalls, [1.0]))
    mpre = np.concatenate(([0.0], precisions, [0.0]))

    for i in range(len(mpre)-1, 0, -1):
        mpre[i-1] = max(mpre[i-1], mpre[i])

    indices = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[indices+1]-mrec[indices]) * mpre[indices+1]))

present_classes = sorted({
    gt["class_id"] for items in ground_truth.values() for gt in items
})
iou_thresholds = np.arange(0.50, 0.96, 0.05)

map_results = []
for condition in IMAGE_DIRS:
    aps_50 = []
    aps_50_95 = []

    for class_id in present_classes:
        aps_50.append(compute_ap_for_class(
            predictions[condition], ground_truth, class_id, 0.50
        ))

        class_aps = []
        for iou in iou_thresholds:
            ap = compute_ap_for_class(
                predictions[condition], ground_truth, class_id, float(iou)
            )
            if not np.isnan(ap):
                class_aps.append(ap)

        if class_aps:
            aps_50_95.append(np.mean(class_aps))

    map_results.append({
        "Condition": condition,
        "mAP50": np.nanmean(aps_50),
        "mAP50-95": np.nanmean(aps_50_95)
    })

map_df = pd.DataFrame(map_results)
print(map_df.to_string(index=False))

## 12. Phase 2 — Q30 preprocessing

In [ ]:
Q30_DIR = IMAGE_DIRS["Q30"]
SHARP_DIR = BASE_DIR / "images_q30_sharpen"
DENOISE_DIR = BASE_DIR / "images_q30_denoise"
UPSCALE_DIR = BASE_DIR / "images_q30_upscale"

for directory in [SHARP_DIR, DENOISE_DIR, UPSCALE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for image_path in sorted(Q30_DIR.glob("*.jpg")):
    img = Image.open(image_path).convert("RGB")

    sharpened = img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=150, threshold=3))
    sharpened.save(SHARP_DIR / image_path.name, "JPEG", quality=95)

    img_cv = cv2.imread(str(image_path))
    denoised = cv2.fastNlMeansDenoisingColored(
        img_cv, None, h=5, hColor=5,
        templateWindowSize=7, searchWindowSize=21
    )
    cv2.imwrite(str(DENOISE_DIR / image_path.name), denoised,
                [cv2.IMWRITE_JPEG_QUALITY, 95])

    upscaled = img.resize((img.width*2, img.height*2), Image.Resampling.LANCZOS)
    upscaled.save(UPSCALE_DIR / image_path.name, "JPEG", quality=95)

print("Q30 preprocessing completed.")

## 13. Recovery inference

In [ ]:
RECOVERY_DIRS = {
    "Q30+Sharpen": SHARP_DIR,
    "Q30+Denoise": DENOISE_DIR,
    "Q30+Upscale": UPSCALE_DIR
}

recovery_results = defaultdict(list)

for condition, image_dir in RECOVERY_DIRS.items():
    print(f"Running YOLO on: {condition}")
    for image_path in sorted(image_dir.glob("*.jpg")):
        result = model.predict(source=str(image_path), conf=0.001, verbose=False)[0]
        recovery_results[condition].append({"image": image_path.name, "result": result})

recovery_predictions = extract_predictions(recovery_results)
print("Recovery inference completed.")

## 14. Recovery metrics

In [ ]:
recovery_pr_results = []
for condition in RECOVERY_DIRS:
    TP, FP, FN, precision, recall = calculate_precision_recall(
        recovery_predictions[condition], ground_truth
    )
    recovery_pr_results.append({
        "Condition": condition, "TP": TP, "FP": FP, "FN": FN,
        "Precision": precision, "Recall": recall
    })

recovery_pr_df = pd.DataFrame(recovery_pr_results)

recovery_map_results = []
for condition in RECOVERY_DIRS:
    aps_50, aps_50_95 = [], []

    for class_id in present_classes:
        aps_50.append(compute_ap_for_class(
            recovery_predictions[condition], ground_truth, class_id, 0.50
        ))

        class_aps = []
        for iou in iou_thresholds:
            ap = compute_ap_for_class(
                recovery_predictions[condition], ground_truth, class_id, float(iou)
            )
            if not np.isnan(ap):
                class_aps.append(ap)

        if class_aps:
            aps_50_95.append(np.mean(class_aps))

    recovery_map_results.append({
        "Condition": condition,
        "mAP50": np.nanmean(aps_50),
        "mAP50-95": np.nanmean(aps_50_95)
    })

recovery_map_df = pd.DataFrame(recovery_map_results)

print(recovery_pr_df.to_string(index=False))
print()
print(recovery_map_df.to_string(index=False))

## 15. Final seven-condition table

In [ ]:
baseline_table = pr_df.merge(map_df, on="Condition")
recovery_table = recovery_pr_df.merge(recovery_map_df, on="Condition")

all_results = pd.concat([baseline_table, recovery_table], ignore_index=True)

condition_order = [
    "Original","Q70","Q50","Q30",
    "Q30+Sharpen","Q30+Denoise","Q30+Upscale"
]

all_results["Condition"] = pd.Categorical(
    all_results["Condition"], categories=condition_order, ordered=True
)
all_results = all_results.sort_values("Condition").reset_index(drop=True)

print(all_results.to_string(index=False))

## 16. Recovery relative to Q30

In [ ]:
q30 = all_results[all_results["Condition"] == "Q30"].iloc[0]

recovery_analysis = []
for condition in ["Q30+Sharpen","Q30+Denoise","Q30+Upscale"]:
    row = all_results[all_results["Condition"] == condition].iloc[0]
    recovery_analysis.append({
        "Condition": condition,
        "mAP50_vs_Q30_%": (row["mAP50"]-q30["mAP50"])/q30["mAP50"]*100,
        "mAP50-95_vs_Q30_%": (row["mAP50-95"]-q30["mAP50-95"])/q30["mAP50-95"]*100
    })

recovery_analysis_df = pd.DataFrame(recovery_analysis)
print(recovery_analysis_df.to_string(index=False))

## 17. Per-image sanity analysis

In [ ]:
def per_image_metrics(condition_predictions, ground_truth,
                       conf_threshold=0.25, iou_threshold=0.50):
    rows = []

    for image_name, gt_items in ground_truth.items():
        gt_boxes = [
            {"class_id": x["class_id"], "box": xywhn_to_xyxy(x["box"]), "matched": False}
            for x in gt_items
        ]

        preds = [
            p for p in condition_predictions.get(image_name, [])
            if p["confidence"] >= conf_threshold
        ]
        preds = sorted(preds, key=lambda x: x["confidence"], reverse=True)

        TP = FP = 0

        for pred in preds:
            best_iou = 0.0
            best_gt = None

            for gt in gt_boxes:
                if gt["matched"] or pred["class_id"] != gt["class_id"]:
                    continue

                iou = box_iou(pred["box"], gt["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt = gt

            if best_gt is not None and best_iou >= iou_threshold:
                TP += 1
                best_gt["matched"] = True
            else:
                FP += 1

        FN = sum(not gt["matched"] for gt in gt_boxes)
        precision = TP/(TP+FP) if TP+FP else 0
        recall = TP/(TP+FN) if TP+FN else 0

        rows.append({
            "Image": image_name, "GT": len(gt_boxes),
            "TP": TP, "FP": FP, "FN": FN,
            "Precision": precision, "Recall": recall
        })

    return pd.DataFrame(rows)

per_image_tables = {}
for condition in IMAGE_DIRS:
    per_image_tables[condition] = per_image_metrics(
        predictions[condition], ground_truth
    )
for condition in RECOVERY_DIRS:
    per_image_tables[condition] = per_image_metrics(
        recovery_predictions[condition], ground_truth
    )

summary_rows = []
for condition, df in per_image_tables.items():
    summary_rows.append({
        "Condition": condition,
        "Mean_Precision": df["Precision"].mean(),
        "Std_Precision": df["Precision"].std(),
        "Mean_Recall": df["Recall"].mean(),
        "Std_Recall": df["Recall"].std()
    })

per_image_summary = pd.DataFrame(summary_rows)
print(per_image_summary.to_string(index=False))

## 18. Final plots

In [ ]:
plt.figure(figsize=(11,6))
plt.bar(all_results["Condition"].astype(str), all_results["mAP50"])
plt.ylabel("mAP50")
plt.xlabel("Condition")
plt.title("YOLO mAP50 under JPEG Compression and Preprocessing")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11,6))
plt.bar(all_results["Condition"].astype(str), all_results["mAP50-95"])
plt.ylabel("mAP50-95")
plt.xlabel("Condition")
plt.title("YOLO mAP50-95 under JPEG Compression and Preprocessing")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
x = np.arange(len(all_results))
width = 0.35

plt.figure(figsize=(12,6))
plt.bar(x-width/2, all_results["Precision"], width, label="Precision")
plt.bar(x+width/2, all_results["Recall"], width, label="Recall")
plt.xticks(x, all_results["Condition"].astype(str), rotation=30)
plt.ylabel("Score")
plt.xlabel("Condition")
plt.title("YOLO Precision and Recall across Experimental Conditions")
plt.legend()
plt.tight_layout()
plt.show()

## 19. Export final result tables

In [ ]:
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

all_results.to_csv(RESULTS_DIR / "mini02_all_results.csv", index=False)
recovery_analysis_df.to_csv(RESULTS_DIR / "mini02_recovery_analysis.csv", index=False)
per_image_summary.to_csv(RESULTS_DIR / "mini02_per_image_summary.csv", index=False)

print("Final result tables saved.")

## 20. Final interpretation

**Main finding:** Q30 caused a meaningful decline in object-detection performance.

**Recovery finding:** Denoising produced the strongest recovery among the three tested methods, especially for Recall and mAP50-95, but the recovery was incomplete.

**Negative findings:** Sharpening improved Precision but harmed Recall and overall mAP. Upscaling did not produce a meaningful recovery.

**Scope:** These conclusions apply to the selected 20-image COCO subset, the pretrained YOLO26n detector, and the exact preprocessing settings used in this study.

# Mini Research #2 — COMPLETE